In [ ]:
# part of the code is taken from https://github.com/shenweichen/DeepCTR-Torch and modified for training
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import dask.dataframe as dd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
import random
from sklearn.metrics import roc_auc_score
from torch.optim import Adagrad

# np.typeDict = np.sctypeDict
import pandas as pd
import torch
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names
from deepctr_torch.models import *


In [ ]:
df = pd.read_csv('sampled_data2.csv')

In [ ]:
df['hour'] = df['hour'].astype(str).str.zfill(8)
df['year'] = '20'+df['hour'].str[:2]
df['month'] = df['hour'].str[2:4]
df['day'] = df['hour'].str[4:6]
df['hour_of_day'] = df['hour'].str[6:8]
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
df['day'] = df['day'].astype(int)
df['hour_of_day'] = df['hour_of_day'].astype(int)

In [ ]:
df['site_id_freq'] = df['site_id'].map(df['site_id'].value_counts())
df['site_domain_freq'] = df['site_domain'].map(df['site_domain'].value_counts())
df['app_id_freq'] = df['app_id'].map(df['app_id'].value_counts())
df['app_domain_freq'] = df['app_domain'].map(df['app_domain'].value_counts())
df['device_id_freq'] = df['device_id'].map(df['device_id'].value_counts())
df['device_ip_freq'] = df['device_ip'].map(df['device_ip'].value_counts())
df['device_model_freq'] = df['device_model'].map(df['device_model'].value_counts())

In [ ]:
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
scaler = MinMaxScaler(feature_range=(0,1))  
encoder = LabelEncoder()  

In [ ]:
sparse_features = ['hour', 'day', 'year', 'month', 'hour_of_day',
 'C1',
 'banner_pos',
 'site_id',
 'site_domain',
 'site_category',
 'app_id',
 'app_domain',
 'app_category',
 'device_id',
 'device_ip',
 'device_model',
 'id']
dense_features = ['device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21']

In [ ]:
for feat in sparse_features:
    df[feat] = encoder.fit_transform(df[feat])  
df[dense_features] = scaler.fit_transform(df[dense_features]) 


In [ ]:
unique_dates = df['day'].unique()

# Calculate the index for the split
split_index = int(0.8 * len(unique_dates))

# Split the dates into training and test dates
train_dates = unique_dates[:split_index]
test_dates = unique_dates[split_index:]

# Filter the data based on these dates
train_df = df[df['day'].isin(train_dates)]
test = df[df['day'].isin(test_dates)]

del df

In [ ]:
# Defining target and features
X_train = train_df.drop(columns=['click'])  # Replace 'click' with your target column name
y_train = train_df['click']

X_test = test.drop(columns=['click'])
y_test = test['click']



In [ ]:
del train_df
del test

In [ ]:
# X_train[sparse_features] = X_train[sparse_features].fillna('-1', )
# X_train[dense_features] = X_train[dense_features].fillna(0, )


# X_test[sparse_features] = X_test[sparse_features].fillna('-1', )
# X_test[dense_features] = X_test[dense_features].fillna(0, )

In [ ]:
#  # 1.Label Encoding for sparse features,and do simple Transformation for dense features
# for feat in sparse_features:
#         lbe = LabelEncoder()
#         X_train[feat] = lbe.fit_transform(X_train[feat])
#         X_test[feat] = lbe.fit_transform(X_test[feat])

In [ ]:
# import pandas as pd
# import numpy as np

# # Example: Assume `df` is your training dataset and `target` is the label column.
# num_subsets = len(X_train)//128  # Number of subsets to create

# # Split the dataset by label
# df_1 = X_train[y_train == 1.0]
# df_0 = X_train[y_train == 0.0]

# # Calculate the number of rows per subset
# subset_size_1 = 53*2
# subset_size_0 = 11*2

# # Create subsets
# subsets = []
# y_label = []
# for i in range(num_subsets):
#     start_1 = i * subset_size_1
#     end_1 = (i + 1) * subset_size_1 
    
#     start_0 = i * subset_size_0
#     end_0 = (i + 1) * subset_size_0
#     if end_1 >= len(df_1) or end_0 >= len(df_0):
#         break
#     # Combine proportional parts of label 1 and label 0
#     subset = pd.concat([df_1.iloc[start_1:end_1], df_0.iloc[start_0:end_0]])
#     ones = [1]*(end_1 - start_1)
#     ones.extend([0]*(end_0 - start_0))
#     subset['label'] = ones

#     # subset = subset.sampl[e(frac=1, random_state=42).reset_index(drop=True)  # Shuffle the subset
#     subsets.append(subset)



# # Verify distribution in each subset
# for i, subset in enumerate(subsets):
#     if i%10==0:
#         print(f"Subset {i+1} distribution:\n", subset['label'].value_counts(normalize=True))
#     if i == 5:
#         break


In [ ]:
# combined_df = pd.concat(subsets, axis=0, ignore_index=True)

# X_train = combined_df.drop(columns=['label'])  # Replace 'click' with your target column name
# y_train = combined_df['label']
# del combined_df
# del subsets


In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'


for feat in dense_features:
    X_train[feat] = X_train[feat].astype('float32')
    X_test[feat] = X_test[feat].astype('float32')

# Convert target to float32
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')

In [ ]:
# 2.count #unique features for each sparse field,and record dense feature field name

fixlen_feature_columns = [SparseFeat(feat, vocabulary_size=X_train[feat].max() + 1, embedding_dim=4)
                              for feat in sparse_features] + [DenseFeat(feat, 1, )
                                                              for feat in dense_features]

In [ ]:
dnn_feature_columns = fixlen_feature_columns
linear_feature_columns = fixlen_feature_columns

feature_names = get_feature_names(
        linear_feature_columns + dnn_feature_columns)

In [ ]:
train_model_input = {name: X_train[name] for name in feature_names}
test_model_input = {name: X_test[name] for name in feature_names}

In [ ]:
torch.set_default_dtype(torch.float32)

for name in feature_names:
    train_model_input[name] = torch.tensor(train_model_input[name].values, dtype=torch.float32)
    test_model_input[name] = torch.tensor(test_model_input[name].values, dtype=torch.float32)

y_train_tensor = y_train.values.astype('float32')


In [ ]:
model = DeepFM(linear_feature_columns=linear_feature_columns, dnn_feature_columns=dnn_feature_columns,
                   task='binary',
                   dnn_dropout=0.5, l2_reg_embedding=0.001, l2_reg_linear=0.001, l2_reg_dnn=0.001,
                       device=device)
model.compile('adam', "binary_crossentropy",
                  metrics=["binary_crossentropy", "auc"], )


In [ ]:
%%time
history = model.fit(train_model_input, y_train_tensor, 
                   batch_size=1024, 
                   epochs=3, 
                   verbose=2,
                   validation_split=0.20)





In [ ]:
# Get predictions on test data
y_pred = model.predict(test_model_input, batch_size=1024)
y_pred_binary = (y_pred > 0.5).astype(int)

# Calculate and print test metrics
accuracy = accuracy_score(y_test, y_pred_binary)
roc_auc = roc_auc_score(y_test, y_pred)

print("\nTest Metrics:")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")


## DeepFM Model with more features

In [ ]:
model_v2 = DeepFM(linear_feature_columns=linear_feature_columns, dnn_feature_columns=dnn_feature_columns,
                   task='binary',
                   dnn_dropout=0.5, l2_reg_embedding=0.001, l2_reg_linear=0.001, l2_reg_dnn=0.001,
                       device=device)
model_v2.compile('adam', "binary_crossentropy",
                  metrics=["binary_crossentropy", "auc"], )


In [ ]:
%%time
history_v2 = model_v2.fit(train_model_input, y_train_tensor, 
                   batch_size=1024, 
                   epochs=3, 
                   verbose=2,
                   validation_split=0.20)





In [ ]:
# Get predictions on test data
y_pred = model_v2.predict(test_model_input, batch_size=1024)
y_pred_binary = (y_pred > 0.5).astype(int)

# Calculate and print test metrics
accuracy = accuracy_score(y_test, y_pred_binary)
roc_auc = roc_auc_score(y_test, y_pred)

print("\nTest Metrics:")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")


## DCN Model

In [ ]:
DCN_model = DCN(linear_feature_columns, dnn_feature_columns, task='binary', l2_reg_linear=0.001, l2_reg_embedding=0.001, dnn_dropout=0.5,  device=device)
DCN_model.compile("adam", "binary_crossentropy", metrics=['binary_crossentropy'])




In [ ]:
%%time
DCN_history = DCN_model.fit(train_model_input, y_train_tensor, batch_size=1024, epochs=1, verbose=3, validation_split=0.2)

In [ ]:
# Get predictions on test data
y_pred = DCN_model.predict(test_model_input, batch_size=1024)
y_pred_binary = (y_pred > 0.5).astype(int)

# Calculate and print test metrics
accuracy = accuracy_score(y_test, y_pred_binary)
roc_auc = roc_auc_score(y_test, y_pred)

print("\nTest Metrics:")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")


## DCN Model with more features

In [ ]:
DCN_model_v2 = DCN(linear_feature_columns, dnn_feature_columns, task='binary', l2_reg_linear=0.001, l2_reg_embedding=0.001, dnn_dropout=0.5,  device=device)
DCN_model_v2.compile("adam", "binary_crossentropy", metrics=['binary_crossentropy'])





In [ ]:
%%time
DCN_history_v2  = DCN_model_v2.fit(train_model_input, y_train_tensor, batch_size=1024, epochs=1, verbose=3, validation_split=0.2)


In [ ]:
# Get predictions on test data
y_pred = DCN_model_v2.predict(test_model_input, batch_size=1024)
y_pred_binary = (y_pred > 0.5).astype(int)

# Calculate and print test metrics
accuracy = accuracy_score(y_test, y_pred_binary)
roc_auc = roc_auc_score(y_test, y_pred)

print("\nTest Metrics:")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")
